# Testing of anderson acceleration

In [1]:
import numpy as np
import matplotlib.pyplot as plot

Function:

$f(x) = sin(x) + atan(x)$ 

$x_0 = 1$ 

In [2]:
# Solution to fixed poiint equation:
x = 1
diff: float = None

for i in range(100):
  x_old = x
  x = np.sin(x) + np.atan(x)
  print((x-x_old))
  g = x-x_old
  if abs(g) < 1e-6:
    break
  print(f"i={i} | x={x} | g={g}")

0.6268691482053448
i=0 | x=1.6268691482053448 | g=0.6268691482053448
0.3912135133447141
i=1 | x=2.018082661550059 | g=0.3912135133447141
-0.005719261317673752
i=2 | x=2.012363400232385 | g=-0.005719261317673752
0.0013288986889894439
i=3 | x=2.0136922989213746 | g=0.0013288986889894439
-0.0003056824953695525
i=4 | x=2.013386616426005 | g=-0.0003056824953695525
7.04812085250417e-05
i=5 | x=2.01345709763453 | g=7.04812085250417e-05
-1.624205504269227e-05
i=6 | x=2.0134408555794874 | g=-1.624205504269227e-05
3.743370829312198e-06
i=7 | x=2.0134445989503167 | g=3.743370829312198e-06
-8.627246912418229e-07


# Simple example of Anderson acceleration about a fixed point function

In [3]:
import numpy as np

x0 = 1.0
k_max = 100
tol_res = 1e-6
m = 3


solve = lambda x: np.sin(x) + np.arctan(x)

# Vector of iterates x and residuals g
x = [float(x0), float(solve(x0))] # NOTE: IC here and SOLVE here as well.
g = [x[1] - x[0], 
     solve(x[1]) - x[1]] # NOTE: SOLVE occurs here as well

# "Matrices" of increments (for scalar: row vectors)
G_k = np.array([[g[1] - g[0]]], dtype=float)  # shape (1, 1)
X_k = np.array([[x[1] - x[0]]], dtype=float)  # shape (1, 1)

k = 2  # corresponds to MATLAB's k=2 (1-based). Here g[k-1] is g(k).
while k < k_max and abs(g[k-1]) > tol_res:
  m_k = min(k, m)

  # Solve the optimization problem by QR decomposition: gamma_k = R \ (Q' * g(k))
  # Here g(k) is scalar; shapes: Q:(1,1), R:(1,1) initially, but can be (1, p).
  Q, R = np.linalg.qr(G_k)  # G_k is (1, p)
  rhs = Q.T @ np.array([[g[k-1]]], dtype=float)  # (p,1) or (1,1) depending on QR
  gamma_k = np.linalg.lstsq(R, rhs, rcond=None)[0]  # robust backsolve, shape (p,1)

  # Compute new iterate and new residual:
  # x(k+1) = x(k) + g(k) - (X_k + G_k) * gamma_k;
  x_next = x[k-1] + g[k-1] - float((X_k + G_k) @ gamma_k)
  g_next = float(solve(x_next) - x_next) # NOTE: PERORM SOLVE

  x.append(x_next)
  g.append(g_next)

  # Update increment matrices with new elements
  X_k = np.hstack([X_k, np.array([[x[k] - x[k-1]]], dtype=float)])
  G_k = np.hstack([G_k, np.array([[g[k] - g[k-1]]], dtype=float)])

  # Keep only the last m_k columns
  ncols = X_k.shape[1]
  if ncols > m_k:
    X_k = X_k[:, ncols - m_k : ]
    G_k = G_k[:, ncols - m_k : ]

  k += 1

# Run the example
print(f"Computed fixed point {x[-1]:.16f} after {k} iterations")

Computed fixed point 2.0134436441742545 after 9 iterations


/var/folders/gt/15kcq_g95yv3l_cjch6ybnzh0000gp/T/ipykernel_27256/2681753517.py:32: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  x_next = x[k-1] + g[k-1] - float((X_k + G_k) @ gamma_k)


# Anderson acceleration of a vector

In [4]:
import numpy as np

x0 = np.random.rand(10)*(1.5-0.5) + 0.5
k_max = 100
tol_res = 1e-6
m=3
norm_order=2

# Function f(x)
solve = lambda x: np.sin(x) + np.arctan(x)

"""
Vector-capable version of the Wikipedia/MATLAB Anderson acceleration code.

- x can be scalar or vector.
- f must return same shape as x (for vectors, elementwise maps are fine).
- Stops when ||g_k|| <= tol_res, where g_k = f(x_k) - x_k.
"""

x0 = np.asarray(x0, dtype=float)
scalar: bool = (x0.ndim == 0)
if scalar:
  x0 = x0.reshape(1)  # treat scalar as 1-vector internally

d = x0.size

# Cache f(x) values to avoid duplicate evaluations
fx0 = np.asarray(solve(x0), dtype=float).reshape(-1) # NOTE: SOLVE
if fx0.size != d:
  raise ValueError(f"f(x0) returned size {fx0.size}, expected {d}")

x = [x0.copy(), fx0.copy()]          # X0 initial condition and the first solution
fx = [fx0.copy(), np.asarray(solve(x[1]), dtype=float).reshape(-1)] # NOTE: SOLVE again
g  = [x[1] - x[0], fx[1] - x[1]]     # g0 = f(x0)-x0 = x1-x0, g1 = f(x1)-x1

# Increment matrices (d x p)
G_k = (g[1] - g[0]).reshape(d, 1)
X_k = (x[1] - x[0]).reshape(d, 1)

k = 2  # MATLAB-like; current residual is g[k-1]
keepGoing = True
while k < k_max and keepGoing:
  m_k = min(k, m)

  # Solve least squares: min || G_k gamma - g_k ||_2
  Q, R = np.linalg.qr(G_k, mode='reduced')      # Q:(d,p), R:(p,p)
  rhs = Q.T @ g[k-1].reshape(d, 1)              # (p,1)
  gamma_k = np.linalg.lstsq(R, rhs, rcond=None)[0]  # (p,1)

  # Wikipedia update
  x_next = x[k-1] + g[k-1] - ((X_k + G_k) @ gamma_k).reshape(d)
  fx_next = np.asarray(solve(x_next), dtype=float).reshape(-1) # this is the solve function.
  if fx_next.size != d:
    raise ValueError(f"f(x) returned size {fx_next.size}, expected {d}")
  g_next = fx_next - x_next

  x.append(x_next)
  fx.append(fx_next)
  g.append(g_next)

  # Update increment matrices
  X_k = np.hstack([X_k, (x[k] - x[k-1]).reshape(d, 1)])
  G_k = np.hstack([G_k, (g[k] - g[k-1]).reshape(d, 1)])

  # Keep only last m_k columns
  ncols = X_k.shape[1]
  if ncols > m_k:
    X_k = X_k[:, ncols - m_k:]
    G_k = G_k[:, ncols - m_k:]

  # Convergence criteria
  nrm = np.linalg.norm(g[k], ord=2)
  if abs(nrm) < tol_res:
    keepGoing = False

  k += 1
  

x_star = x[-1]
if scalar:
  x_star =  float(x_star[0])

# ---- Example: your f(x), but vector IC ----
f = lambda x: np.sin(x) + np.arctan(x)  # works elementwise for NumPy arrays

print("Computed fixed point:", x_star)
print("Iterations:", k)

Computed fixed point: [2.01344386 2.01344388 2.01344405 2.01344383 2.01344387 2.013444
 2.0134439  2.01344386 2.01344386 2.01344388]
Iterations: 12
